In [1]:
import sys
sys.path.append(r'N:\DE\CC Transaction Pipeline')
import cfg.config as config

from pyspark.sql import SparkSession
import os

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType
import pyspark.sql.functions as F

os.environ['HADOOP_HOME'] = r'C:\Users\nisha\hadoop'

spark = SparkSession.builder \
    .appName("CCPipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.367") \
    .config("spark.hadoop.fs.s3a.access.key", config.ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", config.ACCESS_KEY_SECRET) \
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()


In [2]:
slv_fact_transactions = spark.read \
        .format('parquet') \
        .load('s3a://cc-transaction-pipeline/brz/brz_fact_transactions/')

slv_fact_transactions.show()

+---------------------+----------------+--------------------+-------------+------+---------+---------+------+--------------------+----------------+-----+-----+-------+---------+--------+--------------------+----------+--------------------+----------+------------------+------------------+--------+--------------------+--------------------+----+-----+---+
|trans_date_trans_time|          cc_num|            merchant|     category|   amt|    first|     last|gender|              street|            city|state|  zip|    lat|     long|city_pop|                 job|       dob|           trans_num| unix_time|         merch_lat|        merch_long|is_fraud|          merch_uuid|           cust_uuid|year|month|day|
+---------------------+----------------+--------------------+-------------+------+---------+---------+------+--------------------+----------------+-----+-----+-------+---------+--------+--------------------+----------+--------------------+----------+------------------+------------------+--

In [3]:
slv_fact_transactions.count()

1852394

In [11]:
slv_fact_transactions.dropDuplicates().count()

1852394

In [10]:
slv_fact_transactions = slv_fact_transactions[['trans_num', 'merch_uuid', 'cust_uuid', 'trans_date_trans_time', 'cc_num', 'category', 'amt', 'unix_time', 'is_fraud']]

In [13]:
slv_fact_transactions.show()

+--------------------+--------------------+--------------------+---------------------+----------------+-------------+------+----------+--------+
|           trans_num|          merch_uuid|           cust_uuid|trans_date_trans_time|          cc_num|     category|   amt| unix_time|is_fraud|
+--------------------+--------------------+--------------------+---------------------+----------------+-------------+------+----------+--------+
|425b649ca83510dd4...|f02cb284-6e86-4b7...|3666c3ea-d53c-46e...|  2020-12-07 00:00:16|2252055259910912|     misc_net| 69.22|1386374416|       0|
|8898d72f8e0523683...|37812661-853e-4bf...|cc9c27b3-2614-4dc...|  2020-12-07 00:01:00|    568279015842|  grocery_pos|101.06|1386374460|       0|
|b4f038877082f30a0...|63d83f0d-d0ef-428...|f1b66cf0-f651-495...|  2020-12-07 00:01:38|6595970453799027|gas_transport|  66.1|1386374498|       0|
|87e8bcf97a9fc896c...|632a8547-8aae-461...|9ed4913f-567d-434...|  2020-12-07 00:02:31|    630484879808|  grocery_pos| 99.07|138637

In [12]:
slv_fact_transactions.printSchema()

root
 |-- trans_num: string (nullable = true)
 |-- merch_uuid: string (nullable = true)
 |-- cust_uuid: string (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [14]:
slv_fact_transactions = slv_fact_transactions.withColumn('cc_num',F.col('cc_num').cast(StringType()))

In [15]:
slv_fact_transactions.printSchema()

root
 |-- trans_num: string (nullable = true)
 |-- merch_uuid: string (nullable = true)
 |-- cust_uuid: string (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [16]:
slv_fact_transactions.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in slv_fact_transactions.columns]).show()

+---------+----------+---------+---------------------+------+--------+---+---------+--------+
|trans_num|merch_uuid|cust_uuid|trans_date_trans_time|cc_num|category|amt|unix_time|is_fraud|
+---------+----------+---------+---------------------+------+--------+---+---------+--------+
|        0|         0|        0|                    0|     0|       0|  0|        0|       0|
+---------+----------+---------+---------------------+------+--------+---+---------+--------+



In [17]:
slv_fact_transactions.select('category').distinct().show(truncate=False)

+--------------+
|category      |
+--------------+
|travel        |
|misc_net      |
|grocery_pos   |
|kids_pets     |
|shopping_net  |
|grocery_net   |
|food_dining   |
|gas_transport |
|personal_care |
|health_fitness|
|entertainment |
|home          |
|misc_pos      |
|shopping_pos  |
+--------------+



In [18]:
slv_fact_transactions.select('cust_uuid').dropDuplicates().count()

999

In [19]:
slv_fact_transactions.select('merch_uuid').dropDuplicates().count()

693

In [20]:
slv_fact_transactions.write \
    .format('parquet') \
    .mode('overwrite') \
    .save('s3a://cc-transaction-pipeline/slv/slv_fact_transactions/')